# Module 06-2: with_structured_output() 체인 구성

## 학습 목표
- `with_structured_output()`의 동작 원리를 이해할 수 있다
- `CodeReviewResult` 스키마를 정의하고 `structured_llm`을 생성할 수 있다
- FakeLLM 대신 실제 체인 구조로 테스트할 수 있다

**참조 문서**: `docs/part3-prompt-and-llm/06-structured-output.md` 섹션 2.2

## 개념 요약

### with_structured_output() 동작 원리

```
기존 방식:
  프롬프트("JSON 형식으로 답해줘") -> LLM -> 자유 텍스트 -> 파서로 JSON 추출 (실패 가능)

with_structured_output 방식:
  Pydantic 모델 정의 -> LLM이 스키마에 맞게 생성 -> Pydantic 인스턴스 반환 (구조 보장)
```

### 체인 구성 패턴

```python
# 1) 스키마 정의
class MySchema(BaseModel):
    ...

# 2) structured LLM 생성
structured_llm = llm.with_structured_output(MySchema)

# 3) 체인 구성 (StrOutputParser 불필요!)
chain = prompt | structured_llm

# 4) 실행 -> MySchema 인스턴스 반환
result = chain.invoke({...})
print(type(result))  # <class 'MySchema'>
```

> **핵심**: `with_structured_output()`을 사용하면 프롬프트에 JSON 스키마 텍스트를 적을 필요가 없습니다.

In [ ]:
# 환경 설정
import sys
sys.path.insert(0, '..')

import json
from pydantic import BaseModel, Field, ValidationError
from enum import Enum
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from common.fake_llm import FakeLLM

print("환경 설정 완료")

## 실습 1: with_structured_output() 시뮬레이션

FakeLLM은 `with_structured_output()`을 지원하지 않으므로, 체인 구조를 이해하기 위한
시뮬레이션을 구현합니다. 실제 `with_structured_output()`과 동등한 역할을 하는
`StructuredOutputSimulator`를 사용합니다.

In [ ]:
# 1-1. Structured Output 시뮬레이터 구현
class StructuredOutputSimulator:
    """with_structured_output()을 시뮬레이션하는 래퍼.
    
    FakeLLM의 JSON 응답을 Pydantic 모델로 변환합니다.
    실제 LLM에서는 llm.with_structured_output(Schema)로 대체됩니다.
    """
    def __init__(self, llm, schema: type[BaseModel]):
        self.llm = llm
        self.schema = schema
    
    def invoke(self, messages):
        """LLM을 호출하고 결과를 Pydantic 모델로 변환한다."""
        raw_result = self.llm.invoke(messages)
        json_str = raw_result.content if hasattr(raw_result, 'content') else str(raw_result)
        data = json.loads(json_str)
        return self.schema.model_validate(data)
    
    def __ror__(self, prompt):
        """| 연산자 지원."""
        from langchain_core.runnables import RunnableLambda
        
        def run(inputs):
            messages = prompt.format_messages(**inputs)
            return self.invoke(messages)
        
        return RunnableLambda(run)

print("StructuredOutputSimulator 정의 완료")

## 실습 2 (TODO): CodeReviewResult 스키마 정의 → structured_llm 생성 → 체인 실행

`CodeReviewResult` 스키마를 정의하고, FakeLLM + StructuredOutputSimulator로 체인을 구성합니다.

In [ ]:
# TODO: 여기에 코드를 작성하세요
#
# 힌트 1 (방향): 먼저 Severity Enum, Bug 모델, CodeReviewResult 모델을 정의합니다.
#               그 다음 FakeLLM과 StructuredOutputSimulator로 체인을 구성합니다.
#
# 힌트 2 (키워드): class Severity(str, Enum), class Bug(BaseModel), class CodeReviewResult(BaseModel)
#               StructuredOutputSimulator(llm, CodeReviewResult)
#
# 힌트 3 (거의 정답):
#   class Severity(str, Enum):
#       HIGH = "HIGH"
#       MEDIUM = "MEDIUM"
#       LOW = "LOW"
#
#   class Bug(BaseModel):
#       line: int = Field(ge=1, description="버그가 있는 라인 번호")
#       issue: str = Field(description="버그 설명")
#       severity: Severity = Field(description="심각도")
#
#   class CodeReviewResult(BaseModel):
#       bugs: list[Bug] = Field(default_factory=list, description="발견된 버그 목록")
#       score: int = Field(ge=1, le=10, description="코드 품질 점수")
#       summary: str = Field(description="전체 리뷰 요약")

# 2-1. 스키마 정의
class Severity(str, Enum):
    pass  # TODO: HIGH, MEDIUM, LOW

class Bug(BaseModel):
    pass  # TODO: line, issue, severity 필드

class CodeReviewResult(BaseModel):
    """코드 리뷰 결과 스키마."""
    pass  # TODO: bugs, score, summary 필드

In [ ]:
# 2-2. FakeLLM 설정 (JSON 응답 반환)
# TODO: FakeLLM을 생성하고 structured_llm을 만드세요
#
# 힌트: FakeLLM은 JSON 문자열을 반환해야 합니다.
#   review_llm = FakeLLM(responses={
#       "divide": json.dumps({
#           "bugs": [{"line": 2, "issue": "ZeroDivisionError when b is 0", "severity": "HIGH"}],
#           "score": 4,
#           "summary": "나누기 함수에서 0으로 나누는 경우를 처리하지 않습니다."
#       })
#   })
#   structured_llm = StructuredOutputSimulator(review_llm, CodeReviewResult)

review_llm = None   # TODO
structured_llm = None  # TODO

In [ ]:
# 2-3. 프롬프트 + 체인 구성
# TODO: 프롬프트를 정의하고 체인을 구성하세요
#
# 힌트:
#   prompt = ChatPromptTemplate.from_messages([
#       ("system", "당신은 코드 리뷰 전문가입니다.\n주어진 코드의 버그를 분석하세요."),
#       ("human", "언어: {language}\n코드:\n{code}"),
#   ])
#   chain = prompt | structured_llm  # StrOutputParser 불필요!

prompt = None  # TODO
chain = None   # TODO

In [ ]:
# 2-4. 체인 실행
if chain:
    result = chain.invoke({
        "language": "Python",
        "code": "def divide(a, b):\n    return a / b"
    })
    
    print(f"결과 타입: {type(result)}")
    print(f"발견된 버그 수: {len(result.bugs)}")
    for bug in result.bugs:
        print(f"  [{bug.severity}] 라인 {bug.line}: {bug.issue}")
    print(f"품질 점수: {result.score}/10")
    print(f"요약: {result.summary}")

In [ ]:
# 검증 셀
assert review_llm is not None, "review_llm이 None입니다. FakeLLM을 생성하세요."
assert structured_llm is not None, "structured_llm이 None입니다. StructuredOutputSimulator를 생성하세요."
assert prompt is not None, "prompt가 None입니다. ChatPromptTemplate을 생성하세요."
assert chain is not None, "chain이 None입니다. prompt | structured_llm으로 체인을 구성하세요."

# 스키마 검증
assert hasattr(CodeReviewResult, '__fields__') or hasattr(CodeReviewResult, 'model_fields'), "CodeReviewResult가 Pydantic 모델이어야 합니다."

# 실행 결과 검증
test_result = chain.invoke({
    "language": "Python",
    "code": "def divide(a, b):\n    return a / b"
})
assert isinstance(test_result, CodeReviewResult), f"결과가 CodeReviewResult 인스턴스여야 합니다. 현재: {type(test_result)}"
assert hasattr(test_result, 'bugs'), "결과에 bugs 필드가 있어야 합니다."
assert hasattr(test_result, 'score'), "결과에 score 필드가 있어야 합니다."
assert hasattr(test_result, 'summary'), "결과에 summary 필드가 있어야 합니다."
print("실습 2 완료!")

## 실습 3: 실제 with_structured_output() 패턴 이해

실제 LLM(Claude/GPT)을 사용할 때의 코드 패턴을 확인합니다.
(API 키가 없어도 패턴을 이해할 수 있습니다)

In [ ]:
# 실제 LLM을 사용할 때의 패턴 (참고용 - 실행하지 않음)
REAL_LLM_PATTERN = """
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate

# 1) 스키마 정의 (위에서 정의한 CodeReviewResult 사용)

# 2) LLM 생성
llm = ChatAnthropic(model="claude-sonnet-4-5-20250514", temperature=0.1)

# 3) with_structured_output() 적용
structured_llm = llm.with_structured_output(CodeReviewResult)

# 4) 프롬프트 구성 (JSON 스키마 텍스트 불필요!)
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 코드 리뷰 전문가입니다."),
    ("human", "언어: {language}\n코드:\n{code}"),
])

# 5) 체인 실행 (StrOutputParser 불필요!)
chain = prompt | structured_llm
result = chain.invoke({"language": "Python", "code": "def divide(a, b):\n    return a / b"})

# 6) Pydantic 인스턴스로 접근
print(type(result))   # <class 'CodeReviewResult'>
print(result.bugs)    # [Bug(line=2, issue="ZeroDivisionError...", severity=<Severity.HIGH>)]
print(result.score)   # 4
"""

print("실제 LLM 사용 패턴 (참고):")
print(REAL_LLM_PATTERN)

In [ ]:
# 검증 셀
# 이 셀은 이미 위에서 구현된 체인이 올바르게 작동함을 최종 확인합니다
final_result = chain.invoke({
    "language": "Python",
    "code": "def divide(a, b):\n    return a / b"
})

assert isinstance(final_result, CodeReviewResult)
assert 1 <= final_result.score <= 10, f"score는 1~10 범위여야 합니다. 현재: {final_result.score}"
print("실습 3 완료!")

## 정리

이번 실습에서 학습한 내용:

1. **with_structured_output()**: LLM이 Pydantic 스키마에 맞는 출력을 생성하도록 강제
2. **체인 패턴**: `prompt | structured_llm` (StrOutputParser 불필요)
3. **결과 타입**: 문자열이 아닌 Pydantic 인스턴스로 반환됨
4. **FakeLLM 한계**: 실제 `with_structured_output()`은 Anthropic/OpenAI 등 실제 LLM에서만 지원

**다음 실습**: `03_json_parser.ipynb` - JSON 파서 폴백 전략